# JAX-CrossCat Test Runner

Run the full test suite on Kaggle with P100 GPU.

**Instructions:**
1. Upload this notebook to **Kaggle** and select P100 accelerator
2. Set `BRANCH` below to the branch you want to test
3. Run all cells — results are printed inline

## 1. Setup — Install from GitHub

In [ ]:
import os

WORKDIR = "/kaggle/working/jaxcross"

# Branch to test (change as needed)
BRANCH = "main"  # @param {type:"string"}

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git pull)
os.chdir(WORKDIR)

!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

# Preserve Kaggle's pre-installed JAX+CUDA stack to avoid ptxas version mismatch.
# Install jaxcross in editable mode without deps, then add test deps separately.
%pip install -e . --no-deps -q
%pip install pytest pytest-timeout numpyro blackjax ruff -q

print(f"Branch: {BRANCH}")
print(f"Working directory: {os.getcwd()}")
print("Setup complete.")

## 2. Verify GPU/TPU is available

In [ ]:
import jax

print(f"JAX version: {jax.__version__}")
print(f"Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

assert jax.default_backend() == "gpu", "This notebook requires Kaggle P100 GPU runtime!"

import crosscat
print(f"jax-crosscat version: {crosscat.__version__}")

## 3. Run fast tests (excludes slow integration tests)

In [ ]:
!python -m pytest -m "not slow" -v --tb=short 2>&1

## 4. Run full suite (including slow integration/recovery tests)

⚠️ This takes ~15-30 min on a T4 GPU. Skip if you only need fast tests.

In [ ]:
# Set timeout to 600s per test (10 min) — some recovery tests are slow
!python -m pytest -m slow -v --tb=short --timeout=600 2>&1

## 5. Lint check

In [ ]:
!ruff check . 2>&1 && echo "Lint passed" || echo "Lint failed"
!ruff format --check . 2>&1 && echo "Format OK" || echo "Format issues"

## 6. Quick smoke test — end-to-end inference

Verifies core functionality: initialize, packed sweep, queries.

In [ ]:
import time

import jax
import jax.numpy as jnp

from crosscat import (
    column_partition_ari,
    dependence_matrix,
    generate_crosscat_data,
    impute_and_confidence,
    initialize,
    log_joint,
    pack_state,
    packed_gibbs_sweep,
    predictive_anomalousness,
    predictive_sample,
    unpack_state,
)
from crosscat.packed.aot_cache import enable_xla_cache
from crosscat.types import ColumnType

enable_xla_cache()

# Generate synthetic data with known structure
key = jax.random.key(42)
result = generate_crosscat_data(
    key,
    n_rows=200,
    column_types=[
        ColumnType.CONTINUOUS,
        ColumnType.CONTINUOUS,
        ColumnType.CATEGORICAL,
        ColumnType.BINARY,
    ],
    n_views=2,
    n_clusters=3,
)
data = result["data"]
col_types = result["column_types"]
true_col_assigns = result["true_column_assignments"]

print(f"Data shape: {data.shape}")
print(f"Column types: {col_types}")
print(f"True column assignments: {true_col_assigns}")

In [ ]:
# Initialize and run packed inference
key, k1, k2 = jax.random.split(key, 3)
state = initialize(k1, data, col_types)
packed = pack_state(state, max_views=8, max_clusters=16)

print("Running packed Gibbs sweep (first call triggers JIT compilation)...")
t0 = time.time()
packed = packed_gibbs_sweep(k2, packed, data, n_sweeps=50)
t1 = time.time()
print(f"50 sweeps in {t1 - t0:.1f}s (includes JIT compilation)")

# Second run — should be much faster (compiled)
key, k3 = jax.random.split(key)
t0 = time.time()
packed = packed_gibbs_sweep(k3, packed, data, n_sweeps=50)
t1 = time.time()
print(f"50 more sweeps in {t1 - t0:.1f}s (compiled)")

# Unpack and check recovery
state = unpack_state(packed, col_types, data=data)
score = log_joint(state, data)
ari = column_partition_ari(state, true_col_assigns)
print(f"\nLog joint: {score:.2f}")
print(f"Column partition ARI: {ari:.3f}")
print(f"Discovered {state.n_views} views")

In [ ]:
# Query the posterior
key, k4, k5, k6 = jax.random.split(key, 4)

# Predictive sampling
samples = predictive_sample(k4, state, data, query_cols=[0], n_samples=500)
print(f"Predictive samples for col 0: mean={jnp.mean(samples):.2f}, std={jnp.std(samples):.2f}")

# Anomaly detection
anomaly = predictive_anomalousness(k5, state, data, query_row=0)
print(f"Anomaly score (row 0): {anomaly:.3f}")

# Imputation
value, confidence = impute_and_confidence(k6, state, data, query_col=0)
print(f"Imputed col 0: {value:.2f} (confidence: {confidence:.2f})")

# Dependence matrix (Z-matrix)
z = dependence_matrix([state])
print(f"\nDependence matrix (Z-matrix):\n{z}")

print("\n✅ All smoke tests passed!")